In [ ]:
import os
import torch

print("CUDA 可用:", torch.cuda.is_available())
print("GPU 名称:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "无 GPU")


In [ ]:
!pip install -q open_clip_torch

In [ ]:
%cd /content

!wget -O val2017.zip -q --show-progress http://images.cocodataset.org/zips/val2017.zip
!wget -O annotations_trainval2017.zip -q --show-progress http://images.cocodataset.org/annotations/annotations_trainval2017.zip

!unzip -o -q val2017.zip
!unzip -o -q annotations_trainval2017.zip

print("图片数量:")
!find /content/val2017 -maxdepth 1 -name "*.jpg" | wc -l

print("标注文件:")
!ls -lh /content/annotations/captions_val2017.json

In [ ]:
import json
import os
import random

ann_path = "/content/annotations/captions_val2017.json"

with open(ann_path, "r", encoding="utf-8") as f:
    coco_data = json.load(f)

images_by_id = {
    img["id"]: img
    for img in coco_data["images"]
}

captions_by_image = {}
for ann in coco_data["annotations"]:
    captions_by_image.setdefault(
        ann["image_id"], []
    ).append(ann["caption"])

# 固定随机种子，保证每次选到同一批 1000 张
random.seed(42)
all_image_ids = sorted(captions_by_image.keys())
selected_1000_ids = sorted(random.sample(all_image_ids, 1000))

valid_ids = []
missing = []

for image_id in selected_1000_ids:
    image_info = images_by_id[image_id]
    image_path = f"/content/val2017/{image_info['file_name']}"

    if os.path.exists(image_path):
        valid_ids.append(image_id)
    else:
        missing.append(image_path)

metadata = []
for image_id in valid_ids:
    info = images_by_id[image_id]
    metadata.append({
        "image_id": image_id,
        "file_name": info["file_name"],
        "captions": captions_by_image[image_id],
    })

with open("/content/coco1000_image_ids.json", "w", encoding="utf-8") as f:
    json.dump(valid_ids, f, ensure_ascii=False, indent=2)

with open("/content/coco1000_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("总图片数:", len(all_image_ids))
print("选中图片数:", len(selected_1000_ids))
print("有效图片数:", len(valid_ids))
print("缺失图片数:", len(missing))
print("metadata 已生成")

In [ ]:
import torch
import open_clip

# 有 GPU 用 GPU，没有用 CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# 加载 CLIP
model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32-quickgelu",
    pretrained="openai"
)

# 文本 tokenizer
tokenizer = open_clip.get_tokenizer("ViT-B-32-quickgelu")

model = model.to(device)
model.eval()

print("设备:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("警告：当前是 CPU，1000 张图片会比较慢")

In [ ]:
##from google.colab import files

#files.download("/content/coco1000_image_ids.json")
#files.download("/content/coco1000_metadata.json")

In [ ]:
import os
import json
import torch
from PIL import Image

# 读取固定的 1000 张图片 metadata
with open("/content/coco1000_metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

# 检查图片是否存在
valid_metadata = []
missing_files = []

for item in metadata:
    image_path = f"/content/val2017/{item['file_name']}"
    if os.path.exists(image_path):
        valid_metadata.append(item)
    else:
        missing_files.append(image_path)

print("metadata 数量:", len(metadata))
print("有效图片数量:", len(valid_metadata))
print("缺失图片数量:", len(missing_files))

# 整理描述，并记录每条描述属于哪张图片
caption_texts = []
caption_image_indices = []

for image_index, item in enumerate(valid_metadata):
    for caption in item["captions"]:
        caption_texts.append(caption)
        caption_image_indices.append(image_index)

print("描述数量:", len(caption_texts))

# 保存描述到图片索引的映射
with open("/content/coco1000_caption_image_indices.json", "w", encoding="utf-8") as f:
    json.dump(caption_image_indices, f, ensure_ascii=False)

# 分批编码图片
image_batch_size = 32
image_features_list = []

for start in range(0, len(valid_metadata), image_batch_size):
    end = min(start + image_batch_size, len(valid_metadata))
    batch_items = valid_metadata[start:end]

    batch_images = []
    for item in batch_items:
        image_path = f"/content/val2017/{item['file_name']}"
        image = Image.open(image_path).convert("RGB")
        batch_images.append(preprocess(image))

    image_input = torch.stack(batch_images).to(device)

    with torch.no_grad():
        batch_features = model.encode_image(image_input)
        batch_features = batch_features / batch_features.norm(
            dim=-1,
            keepdim=True
        )

    # 放到 CPU，避免一直占 GPU 显存
    image_features_list.append(batch_features.cpu())

    print(f"图片 batch {start}—{end - 1} 完成")

# 拼接成 [1000, 512]
image_features = torch.cat(image_features_list, dim=0)
print("image_features:", image_features.shape)

# 分批编码描述
text_batch_size = 128
text_features_list = []

for start in range(0, len(caption_texts), text_batch_size):
    end = min(start + text_batch_size, len(caption_texts))
    batch_texts = caption_texts[start:end]

    text_input = tokenizer(batch_texts).to(device)

    with torch.no_grad():
        batch_features = model.encode_text(text_input)
        batch_features = batch_features / batch_features.norm(
            dim=-1,
            keepdim=True
        )

    text_features_list.append(batch_features.cpu())
    print(f"文字 batch {start}—{end - 1} 完成")

# 拼接成 [N_caption, 512]
text_features = torch.cat(text_features_list, dim=0)
print("text_features:", text_features.shape)

# 保存向量，避免下次重复编码
torch.save(image_features, "/content/coco1000_image_features.pt")
torch.save(text_features, "/content/coco1000_text_features.pt")

print("已保存:")
print("/content/coco1000_image_features.pt")
print("/content/coco1000_text_features.pt")

In [ ]:
import csv
# 加载之前保存的图片向量和文字向量
image_features = torch.load(
    "/content/coco1000_image_features.pt",
    map_location="cpu"
).to(device)

text_features = torch.load(
    "/content/coco1000_text_features.pt",
    map_location="cpu"
).to(device)

# 读取“每条描述属于哪张图片”的映射
with open(
    "/content/coco1000_caption_image_indices.json",
    "r",
    encoding="utf-8"
) as f:
    caption_image_indices = json.load(f)

print("image_features:", image_features.shape)
print("text_features:", text_features.shape)
print("描述映射数量:", len(caption_image_indices))

# 计算相似度矩阵
# [1000, 512] @ [512, 5004] = [1000, 5004]
with torch.no_grad():
    similarity = image_features @ text_features.T

print("similarity:", similarity.shape)

# 每条描述对应的图片索引
caption_image_indices_tensor = torch.tensor(
    caption_image_indices,
    device=device
)

num_images = image_features.shape[0]

# 每张图片的正确图片索引
correct_image_indices = torch.arange(
    num_images,
    device=device
).unsqueeze(1)

print("\n========== 1000 张图片 Recall ==========")
print("图片数量:", num_images)
print("候选描述数量:", len(caption_image_indices))

metrics = []

for k in [1, 5, 10]:
    # 每张图片取相似度最高的 k 条描述
    topk_caption_indices = similarity.topk(k, dim=1).indices

    # 查出这些描述属于哪张图片
    topk_image_indices = caption_image_indices_tensor[
        topk_caption_indices
    ]

    # 前 k 条描述里有没有属于正确图片的描述
    correct_any = (
        topk_image_indices == correct_image_indices
    ).any(dim=1)

    recall_k = correct_any.float().mean().item()

    print(f"Recall@{k}: {recall_k:.4f}")
    metrics.append([f"Recall@{k}", recall_k])

# Top-1 错误数量
predicted_caption_indices = similarity.argmax(dim=1)
predicted_image_indices = caption_image_indices_tensor[
    predicted_caption_indices
]

error_mask = predicted_image_indices != torch.arange(
    num_images,
    device=device
)

error_count = error_mask.sum().item()
print("Top-1 错误数量:", error_count)

# 保存指标
with open(
    "/content/coco1000_baseline_metrics.csv",
    "w",
    newline="",
    encoding="utf-8"
) as f:
    writer = csv.writer(f)
    writer.writerow(["metric", "value"])
    writer.writerows(metrics)
    writer.writerow(["Top-1 error count", error_count])

print("\n已保存:")
print("/content/coco1000_baseline_metrics.csv")

In [ ]:
from google.colab import files

files.download("/content/coco1000_baseline_metrics.csv")